<div style="border-left: 6px solid #00356B; padding-left: 15px; margin-bottom: 20px;">
  <h1 style="margin-bottom: 5px; color: #00356B"><strong>Assignment 3:</strong> Part 2 (Self-Play with One Model)</h1>
  <span style="font-size: 1.2em; color: #444; font-weight: bold">S&DS 5350 | Social Algorithms</span>
  <br><br>
  <strong>Primary:</strong> Cailey Bobadilla (cjb239)
  <br>
  <strong>Partner:</strong> Brandon Tran (bat53)
  <br>
  <strong>Group:</strong> 9
</div>

---

*Mood for this part:*

<iframe data-testid="embed-iframe" style="border-radius:12px" src="https://open.spotify.com/embed/track/1DwscornXpj8fmOmYVlqZt?utm_source=generator" width="40%" height="152" frameBorder="0" allowfullscreen="" allow="autoplay; clipboard-write; encrypted-media; fullscreen; picture-in-picture" loading="lazy"></iframe>

#### AI Acknowledgement

I used Gemini 3 Pro to develop this Jupyter notebook. My main use was implementing my logic for the `analyze_selfplay()` function.

In [55]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


Now let one model play against itself in a 2-player Scattergories game with many questions.

Use the provided question bank:
- `assets/assignment3/scattergories_questions.csv`

#### II.1 Game definition

Use a two-phase process.

##### Phase A: generate answers

1. For each row `(letter, category)` and each round index, have the player output one answer. Write a prompt that wraps around the `category` and `letter` from `scattergories_questions.csv`, and set the temperature informed by your experiences above. Run both instances of the model (player 1 and player 2) with the same prompt and same temperature (optional: explore varying temperature here as well).
2. Write the answers to a CSV file in the required format.
3. Keep this generation step independent from judging.

Phase A was performed using `part2_phaseA.py` and `assignment3_starter.py`. 

Model Parameters:
- Model: `qwen2.5:7b`
- Temperature: 
    - Default: 0.9
    - Optimal: 2.0
- Top-K: 40
- Number of Rounds: 5
- Prompt: "You are playing a game of Scattergories. Name exactly one {category} 
  that starts with the letter {letter}. Output only the answer, with no 
  punctuation, explanation, or conversation."

##### Head of the Answers for Both Players at Temperature = 0.9 (Default)

In [56]:
import pandas as pd

# Read in CSV files for player 1 and player 2 at temperature=0.9
df_default_p1 = pd.read_csv("part2_selfplay_outputs/default_09_Player_1.csv")
df_default_p2 = pd.read_csv("part2_selfplay_outputs/default_09_Player_2.csv")

# Read in CSV files for player 1 and player 2 at temperature=2.0
df_opt_p1 = pd.read_csv("part2_selfplay_outputs/opt_20_Player_1.csv")
df_opt_p2 = pd.read_csv("part2_selfplay_outputs/opt_20_Player_2.csv")

In [57]:
# Display the head of the answers for player 1 at temperature=0.9
df_default_p1.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Antelope,qwen2.5:7b,Player_1_default,0.9,40,baseline
1,Q001,A,Animals,1,Albatross,qwen2.5:7b,Player_1_default,0.9,40,baseline
2,Q001,A,Animals,2,Ape,qwen2.5:7b,Player_1_default,0.9,40,baseline
3,Q001,A,Animals,3,Albatross,qwen2.5:7b,Player_1_default,0.9,40,baseline
4,Q001,A,Animals,4,Antelope,qwen2.5:7b,Player_1_default,0.9,40,baseline


In [58]:
# Display the head of the answers for player 2 at temperature=0.9
df_default_p2.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Albatross,qwen2.5:7b,Player_2_default,0.9,40,baseline
1,Q001,A,Animals,1,Albatross,qwen2.5:7b,Player_2_default,0.9,40,baseline
2,Q001,A,Animals,2,Alpaca,qwen2.5:7b,Player_2_default,0.9,40,baseline
3,Q001,A,Animals,3,Antelope,qwen2.5:7b,Player_2_default,0.9,40,baseline
4,Q001,A,Animals,4,Antelope,qwen2.5:7b,Player_2_default,0.9,40,baseline


##### Head of the Answers for Both Players at Temperature = 2.0 (Optimal)

In [59]:
# Display the head of the answers for player 1 at temperature=2.0
df_opt_p1.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Albatross,qwen2.5:7b,Player_1_opt,2.0,40,baseline
1,Q001,A,Animals,1,Aardvark,qwen2.5:7b,Player_1_opt,2.0,40,baseline
2,Q001,A,Animals,2,Aardvark,qwen2.5:7b,Player_1_opt,2.0,40,baseline
3,Q001,A,Animals,3,Antelope,qwen2.5:7b,Player_1_opt,2.0,40,baseline
4,Q001,A,Animals,4,Albatross,qwen2.5:7b,Player_1_opt,2.0,40,baseline


In [60]:
# Display the head of the answers for player 2 at temperature=2.0
df_opt_p2.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Antelope,qwen2.5:7b,Player_2_opt,2.0,40,baseline
1,Q001,A,Animals,1,Aardvark,qwen2.5:7b,Player_2_opt,2.0,40,baseline
2,Q001,A,Animals,2,Albatross,qwen2.5:7b,Player_2_opt,2.0,40,baseline
3,Q001,A,Animals,3,Alpaca,qwen2.5:7b,Player_2_opt,2.0,40,baseline
4,Q001,A,Animals,4,Antelope,qwen2.5:7b,Player_2_opt,2.0,40,baseline


##### Phase B: judge and score

1. Run `judge.py` on one or more answer files. Reminder that will require your OpenAI API key.
2. The judge script will:
    - call a GPT judge for validity (`yes`/`no`)
    - normalize answers
    - compute points across submitted player files
    - output score CSV
3. Audit quality: randomly sample at least 50 judged examples and manually verify them; report estimated judge error rate.

Phase B was performed using `judge.py` and this jupyter notebook.

Default Temperature (0.9):
- Judge API calls: 209
- Judge cache hits: 430

Optimized Temperature (2.0):
- Judge API calls: 147
- Judge cache hits: 492

##### Audit Quality Sample (50 Rows)

In [80]:
%%script false --no-raise-error

import pandas as pd

# Read in CSV files for the judged rows from both scattergories games
df_default = pd.read_csv("part2_judged_outputs/judged_default.csv")
df_opt = pd.read_csv("part2_judged_outputs/judged_opt.csv")

# Combine the dataframes into one bigger dataframe
df_all_judged = pd.concat([df_default, df_opt])

# Randomly sample 50 rows for manual verification
audit_sample = df_all_judged.sample(n=50, random_state=42)

# Select the necessary columns for manual verification
audit_sample = audit_sample[['letter', 'category', 'answer_norm', 'valid']]

# Add a blank column for manual verification
audit_sample['manual_verification'] = ""

# Save the sample of 50 to a CSV
audit_sample.to_csv("part2_judged_outputs/audit_quality.csv", index=False)

##### Head of the Manually Verified Judged Rows from the Sample

In [81]:
import pandas as pd

# Read in CSV file for the manual verification of the sampled judged rows
df_audit_sample = pd.read_csv("part2_judged_outputs/audit_quality.csv")

# Display the head of the manual verification of the sampled judged rows
df_audit_sample.head()

,letter,category,answer_norm,valid,manual_verification
0,E,Companies,etsy,1,1
1,F,Things in a backpack,flashlight,1,1
2,Z,Things in a zoo,zookeeper,1,1
3,N,Birds,nightjar,1,1
4,K,Superheroes,kirby,0,0


##### Estimated Judge Accuracy and Error Rates

In [82]:
# Create a series of True and False values based on if columns match in value
matches = df_audit_sample['valid'] == df_audit_sample['manual_verification']

# Calculate the proportion of correct matches
accuracy = matches.mean()

# Print the estimated accuracy and error rates
print(f"Estimated Judge Accuracy Rate: {(accuracy * 100):.1f}%")
print(f"Estimated Judge Error Rate: {((1 - accuracy) * 100):.1f}%")

Estimated Judge Accuracy Rate: 92.0%
Estimated Judge Error Rate: 8.0%


#### II.2 Self-play experiments

1. Once you have generation and judging figured out, run repeated rounds for each question (enough rounds for stable estimates (of the expected score) and store generated answers).
2. Run `judge.py` on your generated files to compute game outcomes from self-play.
3. Measure per-question and overall outcomes:
    - Validity rate
    - Average score per player
4. Revisit prompt/temperature choices and see whether you can improve the self-play score.

Part II.2 was performed using `judge.py`, `part2_selfplay.py`, and this jupyter notebook. 

**NOTE**:
- The answers for different temperature choices were generated in Phase A and Phase B in Part II.1. 
- The answers for different prompt choices were generated using `part2_selfplay.py`.
    - Only the temperature with the higher average score per player (temperature=2.0) was used for the obscure prompt answer generation.

##### Game Analysis Function for Self-Play

In [64]:
def analyze_selfplay(df, game_name):
    """
    Calculates the overall validity and overall average score per player in a 
    game. Also, retrieves the top 3 hardest and easiest categories, and 
    calculates their average points and validity.

    Args:
        df (pd.DataFrame): Contains judged rows from the scattergories game.
        game_name (str): Name of the game.
    """
    print(f"\nOVERALL OUTCOMES: {game_name}")

    # Calculate the overall validity and average score
    overall_validity = df['valid'].mean()
    overall_score = df['score'].mean()

    # Print the overall validity and average score
    print(f"Overall Validity Rate: {overall_validity:.1%}")
    print(
        f"Overall Average Score per Player: {overall_score:.1%} points per "
        "answer"
    )

    # Caclulate metrics for each question
    # NOTE: This is done by grouping by the question features and calculating
    # the mean for valid and score
    per_question = (
        df.groupby(['question_id', 'letter', 'category'])
        [['valid', 'score']]
        .mean()
        .reset_index()
    )

    # Get the 3 hardest and 3 easiest questions
    hardest = per_question.sort_values(by='score').head(3)
    easiest = per_question.sort_values(by='score', ascending=False).head(3)

    print("\nTop 3 Hardest Categories (Lowest Score):")

    # Iterate through each row in the hardest questions/categories
    for _, row in hardest.iterrows():
        # Print the letter and category, and their average points and validity
        print(
            f"Letter {row['letter']} - {row['category']}: {row['score']:.2f} "
            f"average points (Validity: {row['valid']:.1%})"
        )

    print("\nTop 3 Easiest Categories (Highest Score):")

    # Iterate through each row in the easiest questions/categories
    for _, row in easiest.iterrows():
        # Print the letter and category, and their average points and validity
        print(
            f"Letter {row['letter']} - {row['category']}: {row['score']:.2f} "
            f"average points (Validity: {row['valid']:.1%})"
        )

##### Game Analysis for Temperature = 0.9 (Default) and Temperature = 2.0 (Optimal)

In [65]:
import pandas as pd

# Read in CSV files for the judged rows from both scattergories games
df_default = pd.read_csv("part2_judged_outputs/judged_default.csv")
df_opt = pd.read_csv("part2_judged_outputs/judged_opt.csv")

# Analyze the games from the two temperatures
analyze_selfplay(df_default, "Default Game (temperature=0.9)")
analyze_selfplay(df_opt, "Optimized Game (temperature=2.0)")


OVERALL OUTCOMES: Default Game (temperature=0.9)
Overall Validity Rate: 76.9%
Overall Average Score per Player: 32.8% points per answer

Top 3 Hardest Categories (Lowest Score):
Letter P - Historical figures: 0.00 average points (Validity: 100.0%)
Letter O - Companies: 0.00 average points (Validity: 100.0%)
Letter N - Things in a hospital: 0.00 average points (Validity: 100.0%)

Top 3 Easiest Categories (Highest Score):
Letter Q - Words in English: 1.00 average points (Validity: 100.0%)
Letter W - Cities: 1.00 average points (Validity: 100.0%)
Letter G - Foods: 1.00 average points (Validity: 100.0%)

OVERALL OUTCOMES: Optimized Game (temperature=2.0)
Overall Validity Rate: 73.1%
Overall Average Score per Player: 43.4% points per answer

Top 3 Hardest Categories (Lowest Score):
Letter F - Fish: 0.00 average points (Validity: 100.0%)
Letter Z - Animals: 0.00 average points (Validity: 100.0%)
Letter C - Countries: 0.00 average points (Validity: 100.0%)

Top 3 Easiest Categories (Highest 

##### Head of the Answers for Both Players Using the Obscure Prompt (Temperature = 2.0)

Model Parameters:
- Model: `qwen2.5:7b`
- Temperature: 2.0
- Top-K: 40
- Number of Rounds: 5
- Prompt: "You are playing Scattergories. Name a highly obscure, rare, but valid {category} that starts with the letter {letter}. Your goal is to choose a correct answer that no other player will think of. Output only the answer, with no punctuation or explanation."

In [66]:
import pandas as pd

# Read in CSV files for player 1 and player 2 for the obscure prompt 
# (temperature=2.0)
df_obscure_p1 = pd.read_csv("part2_selfplay_outputs/opt_20_Player_1_Obscure.csv")
df_obscure_p2 = pd.read_csv("part2_selfplay_outputs/opt_20_Player_2_Obscure.csv")

In [67]:
# Display the head of the answers for player 1 for the obscure prompt
df_obscure_p1.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Aardvark,qwen2.5:7b,Player_1_Obscure_opt,2.0,40,baseline
1,Q001,A,Animals,1,Aardvark,qwen2.5:7b,Player_1_Obscure_opt,2.0,40,baseline
2,Q001,A,Animals,2,Aardvark,qwen2.5:7b,Player_1_Obscure_opt,2.0,40,baseline
3,Q001,A,Animals,3,Aardvark,qwen2.5:7b,Player_1_Obscure_opt,2.0,40,baseline
4,Q001,A,Animals,4,Axolotl,qwen2.5:7b,Player_1_Obscure_opt,2.0,40,baseline


In [68]:
# Display the head of the answers for player 2 for the obscure prompt
df_obscure_p2.head()

,question_id,letter,category,round_idx,answer,model,player_id,temperature,top_k,prompt_id
0,Q001,A,Animals,0,Aardvark,qwen2.5:7b,Player_2_Obscure_opt,2.0,40,baseline
1,Q001,A,Animals,1,Aardvark,qwen2.5:7b,Player_2_Obscure_opt,2.0,40,baseline
2,Q001,A,Animals,2,Aardvark,qwen2.5:7b,Player_2_Obscure_opt,2.0,40,baseline
3,Q001,A,Animals,3,Albatross,qwen2.5:7b,Player_2_Obscure_opt,2.0,40,baseline
4,Q001,A,Animals,4,Albatross,qwen2.5:7b,Player_2_Obscure_opt,2.0,40,baseline


##### Game Analysis for the Obscure Prompt (Temperature = 2.0)

Obscure Prompt: 
- Judge API calls: 513
- Judge cache hits: 125

In [69]:
# Read in the CSV file for the judged rows from the game with the obscure prompt
df_obscure = pd.read_csv("part2_judged_outputs/judged_obscure.csv")

# Analyze the game with the new, obscure prompt (temperature=2.0)
analyze_selfplay(df_obscure, "Obscure Prompt Game (temperature=2.0)")


OVERALL OUTCOMES: Obscure Prompt Game (temperature=2.0)
Overall Validity Rate: 36.9%
Overall Average Score per Player: 28.7% points per answer

Top 3 Hardest Categories (Lowest Score):
Letter Z - Animals: 0.00 average points (Validity: 100.0%)
Letter X - Brands: 0.00 average points (Validity: 0.0%)
Letter B - Fruits: 0.00 average points (Validity: 0.0%)

Top 3 Easiest Categories (Highest Score):
Letter K - Things in a grocery store: 0.90 average points (Validity: 90.0%)
Letter Q - Words in English: 0.80 average points (Validity: 80.0%)
Letter W - Things in a park: 0.80 average points (Validity: 80.0%)
